# TubeCast — Informe final (tercera entrega)

**Equipo:** Diego Eduardo Martínez Sincel · César Adrián Santos · José Manuel García
**Curso:** Proyecto de Ciencia de Datos, ITESO, otoño 2026

Este informe resume el recorrido completo, de los datos a una predicción servida en AWS. El detalle técnico está en los notebooks enlazados en cada sección.

## 1. Problema y alcance

Empezamos queriendo predecir la viralidad de videos de YouTube con métricas tempranas y sentimiento de comentarios. Al trabajar con la YouTube Data API v3 nos topamos con un límite que cambia todo: la API sólo entrega el conteo **actual** de vistas de cada video, no cuántas tenía hace una semana. Sin una serie histórica real no podíamos predecir el futuro de un video sin hacer trampa con datos que todavía no existían.

Así que redefinimos el problema para que lo que medimos y lo que decimos coincidan:

- **Unidad de observación:** una cohorte diaria, es decir, los videos del canal oficial de la MLB publicados en un día *t*.
- **Target:** las vistas acumuladas por esa cohorte en la fecha de descarga *D*.
- **Entradas:** la misma medida para las 14 cohortes anteriores, tomada en la misma descarga.
- **Qué es y qué no es:** un **análisis retrospectivo de cohortes**. Responde "dado cómo les fue a las 14 cohortes anteriores, ¿cuánto esperaríamos que acumulara esta?", todo visto desde el mismo momento *D*. No es un pronóstico en tiempo real y no mide las vistas que el canal recibe en un día.
- **Cohortes inmaduras:** descartamos las de los últimos 30 días antes de *D*, porque todavía no acumulan sus vistas y distorsionaban la parte final de la serie (y, con ella, la partición de test).

Usuario: alguien del equipo de contenido que quiere comparar el desempeño de lo publicado en un día contra los días cercanos (por ejemplo, para detectar cohortes atípicas durante la temporada).

## 2. Correcciones incorporadas

| Observación de entregas anteriores | Qué hicimos |
|---|---|
| El informe no era un JSON válido (marcador `=======` de un conflicto) | Rehicimos este notebook desde cero con una sola narración |
| Definición temporal: `viewCount` actual sumado por fecha de publicación no son "vistas del día" | Redefinimos el problema como análisis retrospectivo de cohortes; quitamos las cohortes inmaduras; el código, la API, la interfaz y los notebooks usan ese lenguaje |
| Se elegía el modelo con `test_rmse` | `task_comparar_modelos` elige por `val_rmse`; test se reporta después sin intervenir |
| MAPE con ceros reemplazados por `1e-8` | El MAPE se calcula sólo en días con vistas > 0 |
| El flow capturaba cualquier error del Registry y decía "finalizado con éxito" | Con Databricks configurado, la task de registro falla si no completa; la metadata guarda `registro_completo` |
| Tracking y Registry en `sqlite:///mlflow.db` | Reejecutamos el flow con `databricks` / `databricks-uc` en un solo workspace (secciones 4 y 5) |
| La API cargaba `@champion` sin fijar la versión y caía en silencio al artefacto local | Resuelve el alias a un número de versión y carga esa versión; el respaldo local sólo se usa con `MODEL_SOURCE=local` y se reporta como tal |
| Interfaz hablaba del "tráfico del día siguiente" | Ahora habla de cohortes y vistas acumuladas en la misma instantánea |
| README desactualizado, enlace `edu_captura3.png` roto, captura duplicada, nombres pendientes | README reescrito; enlace corregido; quitamos `prep_captura3.png`; integrantes y responsables identificados |
| Faltaban capturas de Prefect, MLflow, Registry, API y frontend | Agregadas en `evidencia/` (sección 8) |

Como las correcciones cambian los datos (cohortes inmaduras), la métrica MAPE y la regla de selección, **regeneramos todos los resultados y modelos**: las versiones que se citan abajo son las de la nueva ejecución.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Los notebooks se ejecutan desde la raíz del repositorio para compartir rutas con src/
RAIZ = Path.cwd().parent if Path.cwd().name in {"notebooks", "informe"} else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("Raíz del repositorio:", RAIZ)

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from IPython.display import display, Markdown
pd.options.display.float_format = "{:,.2f}".format

from src.entrenamiento.reportes import configurar_mlflow, runs_ultima_ejecucion, tabla_comparacion, versiones_registradas
cfg = configurar_mlflow()
print("Tracking :", cfg["tracking_uri"], "| Registry:", cfg["registry_uri"])
print("Experimento:", cfg["experimento"])
print("Modelo     :", cfg["modelo"], "@" + cfg["alias"])
assert cfg["tracking_uri"].startswith("databricks"), "El informe debe leer el tracking remoto de Databricks"

## 3. Datos

Descargamos los ~20 000 videos más recientes del canal de la MLB con la YouTube Data API v3 (detalle, diccionario y privacidad en [data/README.md](../data/README.md)). Agrupamos por fecha de publicación, rellenamos con 0 los días sin publicaciones y armamos ventanas de 14 cohortes. Las particiones son temporales, sin mezcla: 70 % entrenamiento, 15 % validación (para el HPO y la selección) y 15 % test (sólo para la evaluación final). Ver [03-experimentos](../notebooks/03-experimentos.ipynb).

In [ ]:
from src.entrenamiento.datos import descargar_datos_mlb, construir_serie_diaria, crear_ventanas_tabulares, particionar_serie, DIAS_MADURACION

df_raw = descargar_datos_mlb()
serie = construir_serie_diaria(df_raw)
X, y, fechas = crear_ventanas_tabulares(serie)
part = particionar_serie(X, y, fechas)
resumen = pd.DataFrame({n: {"filas": len(part[f"X_{n}"]),
                            "desde": part[f"dates_{n}"].min().date(),
                            "hasta": part[f"dates_{n}"].max().date(),
                            "mediana_target": part[f"y_{n}"].median()} for n in ["train", "val", "test"]}).T
print(f"{len(df_raw):,} videos · {len(serie)} cohortes (sin los últimos {DIAS_MADURACION} días)")
display(resumen)

## 4. Modelado y selección

Comparamos tres familias con el mismo presupuesto de búsqueda (Optuna, `TPESampler(seed=42)`, 20 trials cada una) y la misma partición: **Ridge** (lineal regularizado), **Random Forest** (bagging de árboles) e **HistGradientBoosting** (boosting de árboles). El escalado va dentro de cada `Pipeline`, así que se ajusta sólo con entrenamiento y viaja con el modelo hasta la API. El mejor modelo de cada familia se reentrena con train + validación y se evalúa una sola vez en test. Detalle de los trials y rangos en [03-experimentos](../notebooks/03-experimentos.ipynb).

In [ ]:
r = runs_ultima_ejecucion(cfg["experimento"])
comp = tabla_comparacion(r["finales"])
display(comp.drop(columns=["run_id"]))
ganador = comp.iloc[0]
display(Markdown(f"**Seleccionada por menor `val_rmse`: `{ganador['familia']}`** "
                 f"(val_rmse = {ganador['val_rmse']:,.0f}; test_rmse = {ganador['test_rmse']:,.0f}, "
                 f"test_mae = {ganador['test_mae']:,.0f})."))

**Lectura de los resultados.** _(completar después de ejecutar: qué familia ganó, por cuánto frente a las otras y frente al baseline de media móvil de [03-experimentos](../notebooks/03-experimentos.ipynb), y si el orden en validación se mantiene en test.)_ Un RMSE de millones de vistas parece enorme, pero hay que leerlo contra la escala: las cohortes van de cero a decenas de millones según haya partidos o highlights virales, y ese ruido es justo lo que el modelo no puede anticipar sólo con lags.

## 5. Registro y trazabilidad

El flow de Prefect (`tubecast-training-pipeline`, [05-pipeline](../notebooks/05-pipeline.ipynb)) coordina carga y validación de datos, preparación, las tres búsquedas, la comparación y el registro. Los runs quedan en el experimento `/Shared/pcd-otono-2026-proyecto-tubecast` de Databricks y los tres pipelines finales como versiones de `workspace.default.pcd_otono_2026_tubecast`, cada una con su familia, run de origen y métricas. El alias `champion` apunta a la versión elegida. Ver [04-seleccion-registro](../notebooks/04-seleccion-registro.ipynb).

In [ ]:
versiones = versiones_registradas(cfg["modelo"])
ultimas = versiones[versiones["run_id"].isin(comp["run_id"])]
display(ultimas)
champion = ultimas[ultimas["aliases"].str.contains(cfg["alias"])]
assert len(ultimas) == 3 and len(champion) == 1
assert champion.iloc[0]["familia"] == ganador["familia"]
display(Markdown(f"`{cfg['modelo']}@{cfg['alias']}` → **versión {champion.iloc[0]['version']}** "
                 f"({champion.iloc[0]['familia']}). Es la versión que sirve la API."))

## 6. Arquitectura actual

```text
Computadora local                 Databricks                      EC2 (us-east-1)
-----------------                 ----------                      ---------------
YouTube API → datos.py            Experimento MLflow              tubecast-ui-container  :8501 (0.0.0.0)
flow Prefect → HPO 3 familias ──► runs y métricas                        │ HTTP 127.0.0.1:8000 (red host)
                              ──► Unity Catalog: v1, v2, v3       tubecast-api-container :8000 (127.0.0.1)
                                  alias champion ───────────────► carga la versión del alias
```

El mapa completo, con contratos entre componentes, está en [docs/mapa-sistema-ml.md](../docs/mapa-sistema-ml.md).

- **API (FastAPI):** al arrancar resuelve `champion` a una versión concreta y la carga una sola vez; valida que lleguen exactamente 14 valores numéricos, finitos y no negativos (si no, 422); construye las mismas columnas `lag_14 … lag_1` del entrenamiento y responde con la estimación, su interpretación y el nombre, versión, familia y origen del modelo. No reentrena. Ver [06-api](../notebooks/06-api.ipynb).
- **Interfaz (Streamlit):** formulario de 14 cohortes que consulta la API por HTTP y muestra la versión que respondió. No tiene credenciales. Ver [07-frontend](../notebooks/07-frontend.ipynb).

## 7. Contenedores y despliegue

Cada componente tiene su propia imagen con Python 3.12 y las dependencias exportadas de `uv.lock` (`src/api/Dockerfile`, `src/frontend/Dockerfile`). `docker-compose.yaml` levanta los dos servicios con **red host**: los contenedores comparten la red del anfitrión, así que para la interfaz `127.0.0.1:8000` es la API corriendo en el mismo servidor. Los secretos llegan al ejecutar desde un `.env` que no está en Git ni en las imágenes. Ver [08-contenedores](../notebooks/08-contenedores.ipynb).

Desplegamos en una sola instancia EC2 ya asignada a _(completar)_, en `us-east-1`. SSH sólo desde la IP del responsable; Streamlit (8501) sólo desde las IP de quienes prueban; la API no se expone (escucha en `127.0.0.1`). Obtuvimos una predicción desde otra computadora y comprobamos que la versión mostrada coincide con `/health`, con el log de la API y con `champion` en el Registry. Al terminar retiramos los contenedores y detuvimos la instancia; sólo el volumen EBS sigue generando costo. Ver [09-despliegue](../notebooks/09-despliegue.ipynb).

**Predicción demostrada en EC2:** _(completar: entrada, vistas estimadas y versión `vN`)_.

## 8. Evidencias

| Caso | Captura | Qué muestra |
|---|---|---|
| Experimento remoto | [t7-01](../evidencia/t7-01-experimento-remoto.png) | Ruta `/Shared/pcd-otono-2026-proyecto-tubecast` con los runs `hpo-ridge`, `hpo-random-forest`, `hpo-gradient-boosting` |
| HPO por familia | [ridge](../evidencia/t7-02-hpo-ridge.png) · [random-forest](../evidencia/t7-02-hpo-random-forest.png) · [gradient-boosting](../evidencia/t7-02-hpo-gradient-boosting.png) | Trials `trial-NNN` con hiperparámetros, `val_rmse` y mejor resultado |
| Comparación final | [t7-03](../evidencia/t7-03-comparacion.png) | Runs `<familia>-final` con `val_rmse` y métricas de test |
| Model Registry | [t7-04](../evidencia/t7-04-model-registry.png) | Tres versiones con familia y run de origen; `champion` en la elegida |
| Prefect | [t7-05](../evidencia/t7-05-prefect.png) | Flow `tubecast-training-pipeline` Completed con sus cinco tasks |
| API válida / inválida | [t7-06](../evidencia/t7-06-api-valida.png) · [t7-07](../evidencia/t7-07-api-invalida.png) | 200 con predicción y versión; 422 sin predicción |
| Frontend integrado | [t7-08](../evidencia/t7-08-frontend.png) | Formulario, resultado y la petición correspondiente en la API |
| Ambiente y builds | [t8-01](../evidencia/t8-01-uv-sync-builds.png) | `uv sync --locked` y las dos imágenes construidas |
| Integración local | [t8-02](../evidencia/t8-02-integracion-local.png) | Servicios arriba (API `healthy`) y predicción en contenedores |
| Instancia y red | [t8-03](../evidencia/t8-03-ec2-instancia-red.png) | EC2 Running y reglas de entrada 22/8501 restringidas |
| Despliegue | [t8-04](../evidencia/t8-04-ec2-despliegue.png) | `compose up` y `ps` en EC2 |
| Predicción remota | [t8-05](../evidencia/t8-05-prediccion-remota.png) | Formulario desde otra computadora y versión servida |
| Entrada inválida en EC2 | [t8-06](../evidencia/t8-06-ec2-entrada-invalida.png) | 422 por `curl` dentro del servidor |
| Logs | [t8-07](../evidencia/t8-07-logs.png) | Carga del modelo y peticiones con su versión |
| Cierre | [t8-08](../evidencia/t8-08-cierre.png) | `compose down` e instancia `Stopped` |

## 9. Limitaciones

- **Una sola instantánea.** Todas las cohortes se miden en la misma fecha *D*, pero tienen distinta antigüedad (de 30 días a casi dos años). Quitamos las inmaduras, pero no igualamos el horizonte; para eso habría que guardar descargas periódicas y medir cada cohorte a un número fijo de días.
- **No es un pronóstico.** Las entradas incluyen vistas que esos videos ganaron después del día *t*. El modelo sirve para comparar cohortes en retrospectiva, no para anticipar el futuro.
- **Sólo lags.** Los picos por partidos o highlights virales dependen de eventos externos que el modelo no ve; ahí está la mayor parte del error.
- **Un canal.** Los resultados son del canal de la MLB y no se trasladan a otros canales sin reentrenar.
- **Operación manual.** El reentrenamiento se lanza a mano y la instancia se enciende sólo para demostrar.

## 10. Siguientes pasos

1. Programar descargas periódicas para construir snapshots y medir cada cohorte a un horizonte fijo.
2. Agregar variables de calendario y de eventos (días de partido, playoffs).
3. Retomar el análisis de comentarios como variable exógena una vez que haya una serie histórica.

## 11. Referencias y uso de IA

- Google Developers, *YouTube Data API v3*, https://developers.google.com/youtube/v3
- Documentación de MLflow, Optuna, Prefect, FastAPI, Streamlit y Docker Compose; materiales de las clases 10–20 del curso.
- AWS, *Stop and start Amazon EC2 instances*, https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/Stop_Start.html

Usamos un asistente de IA (Claude) como apoyo para revisar código, redactar documentación y preparar Dockerfiles y Compose. Revisamos y ejecutamos todo lo entregado; las cifras de este informe salen de nuestras ejecuciones.